In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd
from shapely.geometry import box
from deepforest import main as deepforest_main

import importlib
ground = importlib.import_module('00_ground_truth_helpers')

In [13]:
PATCH_SIZE = 800
PATCH_OVERLAP = 0.05
IOU_THRESHOLD = 0.15
DATALOADER_STRATEGY = 'batch' # 'window'
GRID_SHAPE_1M = (1000, 1000)

In [14]:
def load_deepforest_model():
    model = deepforest_main.deepforest()
    model.load_model(model_name="weecology/deepforest-tree", revision="main")
    return model

In [15]:
def predict_tile_boxes(model, rgb_path, patch_size, patch_overlap, iou_threshold, dataloader_strategy):
    """
    Run DeepForest predict_tile on a georeferenced RGB tif and return detections as a GeoDataFrame.

    Purpose:
        Split the large RGB tile into overlapping patches, run the pretrained
        detector on each, and reassemble into a single set of non-overlapping
        (post-NMS) bounding boxes in the tile's native CRS.

    Outputs:
        boxes_gdf : GeoDataFrame with columns [tree_id, label, score, geometry], CRS matches rgb_path
    """
    df = model.predict_tile(path=str(rgb_path), patch_size=patch_size, patch_overlap=patch_overlap, iou_threshold=iou_threshold, dataloader_strategy=dataloader_strategy)
    with rasterio.open(rgb_path) as src:
        crs = src.crs
        transform = src.transform
    if 'geometry' in df.columns and isinstance(df, gpd.GeoDataFrame) and df.crs is not None:
        boxes_gdf = df.copy()
    else:
        xmin_geo, ymax_geo = rasterio.transform.xy(transform, df['ymin'], df['xmin'], offset='ul')
        xmax_geo, ymin_geo = rasterio.transform.xy(transform, df['ymax'], df['xmax'], offset='ul')
        geoms = [box(x0, y0, x1, y1) for x0, y0, x1, y1 in zip(xmin_geo, ymin_geo, xmax_geo, ymax_geo)]
        boxes_gdf = gpd.GeoDataFrame(df.copy(), geometry=geoms, crs=crs)
    boxes_gdf = boxes_gdf.reset_index(drop=True)
    boxes_gdf['tree_id'] = boxes_gdf.index + 1
    if 'label' not in boxes_gdf.columns:
        boxes_gdf['label'] = 'Tree'
    boxes_gdf = boxes_gdf[['tree_id', 'label', 'score', 'geometry']]
    return boxes_gdf

In [16]:
def rasterize_score_layer(boxes_gdf, reference_profile, out_shape):
    """
    Rasterize per-box detection scores onto the reference grid.

    Purpose:
        Produce a continuous score surface: pixels outside every bounding box are 0,
        pixels inside one or more boxes take the max score among overlapping boxes.
        rasterio.features.rasterize has no native max merge, so boxes are sorted
        ascending by score and burned with MergeAlg.replace, letting the highest
        score win on any overlap.

]    Outputs:
        score_arr : 2D float32 array, shape out_shape, 0 outside boxes, max score inside
    """
    if len(boxes_gdf) == 0:
        return np.zeros(out_shape, dtype=np.float32)
    ordered = boxes_gdf.sort_values('score', ascending=True)
    shapes_scores = [(geom, float(s)) for geom, s in zip(ordered.geometry, ordered['score'])]
    score_arr = rasterize(shapes_scores, out_shape=out_shape, transform=reference_profile['transform'], fill=0.0, merge_alg=rasterio.enums.MergeAlg.replace, dtype='float32')
    return score_arr

In [17]:
def rasterize_tree_mask(boxes_gdf, reference_profile, out_shape):
    """
    Rasterize bounding boxes into a binary tree mask.

    Outputs:
        mask_arr : uint8 array, shape out_shape, 1 inside any box, 0 elsewhere
    """
    if len(boxes_gdf) == 0:
        return np.zeros(out_shape, dtype=np.uint8)
    shapes_vals = [(geom, 1) for geom in boxes_gdf.geometry]
    mask_arr = rasterize(shapes_vals, out_shape=out_shape, transform=reference_profile['transform'], fill=0, merge_alg=rasterio.enums.MergeAlg.replace, dtype='uint8')
    return mask_arr

In [18]:
def write_geotiff(array, reference_profile, out_path, dtype, nodata):
    """Write a 2D single-band array as GeoTIFF matching reference_profile."""
    prof = reference_profile.copy()
    prof.update(count=1, dtype=dtype, nodata=nodata, compress='lzw', height=array.shape[0], width=array.shape[1])
    with rasterio.open(out_path, 'w', **prof) as dst:
        dst.write(array.astype(dtype), 1)

In [19]:
def process_tile_deepforest(tile_id, model):
    """
    Run DeepForest prediction on one tile's RGB, write crown outlines, score raster, and tree mask.

    Outputs:
        None (writes GeoPackage and GeoTIFF outputs to ground.OUTPUT_DIR)
    """
    rgb_path, _, savi_path, _ = ground.build_paths(tile_id)
    print(f"Tile {tile_id}")
    print(f"RGB: {rgb_path.name}")

    boxes_gdf = predict_tile_boxes(model, rgb_path, PATCH_SIZE, PATCH_OVERLAP, IOU_THRESHOLD, DATALOADER_STRATEGY)
    n_boxes = len(boxes_gdf)
    print(f"detections: {n_boxes}")
    if n_boxes > 0:
        print(f"score min/mean/median/max: {boxes_gdf['score'].min():.3f} / {boxes_gdf['score'].mean():.3f} / {boxes_gdf['score'].median():.3f} / {boxes_gdf['score'].max():.3f}")

    with rasterio.open(savi_path) as src:
        ref_1m_profile = src.profile.copy()

    outlines_path = ground.OUTPUT_DIR / f"tree_crown_outlines_deepforest_{tile_id}_{ground.YEAR}.gpkg"
    boxes_gdf.to_file(outlines_path, driver='GPKG')
    print(f"wrote {outlines_path.name}")

    score_arr = rasterize_score_layer(boxes_gdf, ref_1m_profile, GRID_SHAPE_1M)
    score_path = ground.OUTPUT_DIR / f"tree_score_deepforest_{tile_id}_{ground.YEAR}.tif"
    write_geotiff(score_arr, ref_1m_profile, score_path, 'float32', -1)
    print(f"wrote {score_path.name}")

    mask_arr = rasterize_tree_mask(boxes_gdf, ref_1m_profile, GRID_SHAPE_1M)
    mask_path = ground.OUTPUT_DIR / f"tree_mask_deepforest_{tile_id}_{ground.YEAR}.tif"
    write_geotiff(mask_arr, ref_1m_profile, mask_path, 'uint8', 255)
    print(f"wrote {mask_path.name}")

    n_tree = int(mask_arr.sum())
    n_total = mask_arr.size
    print(f"tree pixel fraction: {n_tree:,} / {n_total:,} ({n_tree / n_total * 100:.2f}%)")

# Run

In [20]:
model = load_deepforest_model()

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [21]:
for tid in ground.TILE_IDS:
    process_tile_deepforest(tid, model)

515000_3530000
Tile 515000_3530000
RGB: 2022_SRER_5_515000_3530000_image.tif


/projectnb/modislc/fache/.conda/envs/LCSC/lib/python3.11/site-packages/PIL/Image.py:3574: DecompressionBombWarning: Image size (100000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Output()

/projectnb/modislc/fache/.conda/envs/LCSC/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


RuntimeError: [enforce fail at alloc_cpu.cpp:127] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 8028160000 bytes. Error code 12 (Cannot allocate memory)